In [21]:
import pandas as pd
import re
import numpy as np

print("="*80)
print("STEP 1: DATA LOADING & MULTICLASS LABELING")
print("="*80)

# Load original dataset
df = pd.read_csv('payload_full.csv')

print(f"\n✓ Dataset loaded: {len(df)} samples")
print(f"\nOriginal label distribution:")
print(df['label'].value_counts())

# Filter only SQL injection attacks
sqli_df = df[df['attack_type'] == 'sqli'].copy()
print(f"\n✓ SQL Injection samples: {len(sqli_df)}")

# ============================================
# CATEGORIZE INTO MULTICLASS LABELS
# ============================================
def categorize_sqli_attack(payload):
    """
    Categorize SQL injection into specific attack types
    """
    if pd.isna(payload):
        return 'unknown'
    
    payload_lower = str(payload).lower()
    
    # 1. UNION-BASED
    if 'union' in payload_lower and 'select' in payload_lower:
        return 'union_based'
    
    # 2. TIME-BASED BLIND
    elif any(keyword in payload_lower for keyword in ['waitfor delay', 'sleep(', 'sleep (', 'benchmark(', 'pg_sleep']):
        return 'time_based_blind'
    
    # 3. BOOLEAN-BASED BLIND
    elif re.search(r"(and|or)\s+\d+\s*[=<>!]+\s*\d+", payload_lower):
        return 'boolean_blind'
    
    # 4. ERROR-BASED
    elif any(func in payload_lower for func in ['convert(', 'cast(', 'concat(', 'extractvalue(', 'updatexml(', 'exp(', '@@version']):
        return 'error_based'
    
    # 5. STACKED QUERIES
    elif ';' in payload and any(cmd in payload_lower for cmd in ['drop', 'insert', 'update', 'delete', 'create', 'alter', 'exec', 'execute']):
        return 'stacked_queries'
    
    # 6. PIGGY-BACKED QUERIES
    elif ';' in payload and 'select' in payload_lower:
        return 'piggy_backed'
    
    # 7. OTHER SQL INJECTION
    elif any(keyword in payload_lower for keyword in ['select', 'union', 'where', 'from', 'order by']):
        return 'other_sqli'
    
    # Unknown
    else:
        return 'unknown'

# Apply categorization
print("\n✓ Categorizing SQL injection attacks...")
sqli_df['sqli_class'] = sqli_df['payload'].apply(categorize_sqli_attack)

print("\nSQL Injection class distribution:")
print(sqli_df['sqli_class'].value_counts())

# Merge small classes into 'other_sqli'
sqli_df['sqli_class'] = sqli_df['sqli_class'].replace({
    'stacked_queries': 'other_sqli',
    'unknown': 'other_sqli'
})

print("\nAfter merging small classes:")
print(sqli_df['sqli_class'].value_counts())

# Add benign samples
benign_df = df[df['label'] == 'norm'].copy()
benign_df['sqli_class'] = 'benign'

# Combine: Benign + SQL Injection attacks
final_df = pd.concat([
    benign_df[['payload', 'sqli_class']], 
    sqli_df[['payload', 'sqli_class']]
], ignore_index=True)

# Shuffle
final_df = final_df.sample(frac=1, random_state=42).reset_index(drop=True)

print("\n" + "="*80)
print("FINAL DATASET:")
print("="*80)
print(f"Total samples: {len(final_df)}")
print("\nClass distribution:")
print(final_df['sqli_class'].value_counts())
print("\nPercentage distribution:")
print((final_df['sqli_class'].value_counts() / len(final_df) * 100).round(2))

# Save
final_df.to_csv('sqli_multiclass_dataset.csv', index=False)
print("\n✓ Saved to 'sqli_multiclass_dataset.csv'")
print("="*80)

STEP 1: DATA LOADING & MULTICLASS LABELING

✓ Dataset loaded: 31067 samples

Original label distribution:
label
norm    19304
anom    11763
Name: count, dtype: int64

✓ SQL Injection samples: 10852

✓ Categorizing SQL injection attacks...

SQL Injection class distribution:
sqli_class
other_sqli          2722
time_based_blind    2039
union_based         1952
unknown             1151
piggy_backed        1136
error_based          980
boolean_blind        794
stacked_queries       78
Name: count, dtype: int64

After merging small classes:
sqli_class
other_sqli          3951
time_based_blind    2039
union_based         1952
piggy_backed        1136
error_based          980
boolean_blind        794
Name: count, dtype: int64

FINAL DATASET:
Total samples: 30156

Class distribution:
sqli_class
benign              19304
other_sqli           3951
time_based_blind     2039
union_based          1952
piggy_backed         1136
error_based           980
boolean_blind         794
Name: count, dtype: i

In [22]:
import pandas as pd
import numpy as np
from collections import Counter

print("="*80)
print("STEP 2: FEATURE EXTRACTION (Ultra-Clean, No Leakage)")
print("="*80)

def extract_ultra_clean_features(payload):
    """
    Extract ONLY generic statistical features
    NO SQL keywords, NO attack-specific patterns
    """
    if pd.isna(payload):
        payload = ""
    
    payload = str(payload)
    
    features = {}
    length = len(payload)
    
    # ============================================
    # 1. CHARACTER FREQUENCY (Generic)
    # ============================================
    features['num_spaces'] = payload.count(' ')
    features['num_single_quotes'] = payload.count("'")
    features['num_double_quotes'] = payload.count('"')
    features['num_dashes'] = payload.count('-')
    features['num_semicolons'] = payload.count(';')
    features['num_parentheses'] = payload.count('(') + payload.count(')')
    features['num_equals'] = payload.count('=')
    features['num_asterisks'] = payload.count('*')
    features['num_percent'] = payload.count('%')
    features['num_commas'] = payload.count(',')
    features['num_slashes'] = payload.count('/') + payload.count('\\')
    features['num_pipes'] = payload.count('|')
    features['num_ampersand'] = payload.count('&')
    features['num_dots'] = payload.count('.')
    features['num_underscores'] = payload.count('_')
    features['num_colons'] = payload.count(':')
    
    # ============================================
    # 2. RATIO FEATURES (Generic Statistics)
    # ============================================
    if length > 0:
        features['uppercase_ratio'] = sum(1 for c in payload if c.isupper()) / length
        features['lowercase_ratio'] = sum(1 for c in payload if c.islower()) / length
        features['digit_ratio'] = sum(1 for c in payload if c.isdigit()) / length
        features['alpha_ratio'] = sum(1 for c in payload if c.isalpha()) / length
        features['special_char_ratio'] = sum(1 for c in payload if not c.isalnum() and c != ' ') / length
        features['whitespace_ratio'] = sum(1 for c in payload if c.isspace()) / length
    else:
        features['uppercase_ratio'] = 0
        features['lowercase_ratio'] = 0
        features['digit_ratio'] = 0
        features['alpha_ratio'] = 0
        features['special_char_ratio'] = 0
        features['whitespace_ratio'] = 0
    
    # ============================================
    # 3. ENTROPY (Randomness)
    # ============================================
    if length > 0:
        counter = Counter(payload)
        entropy = -sum((count/length) * np.log2(count/length) 
                      for count in counter.values())
        features['entropy'] = entropy
    else:
        features['entropy'] = 0
    
    # ============================================
    # 4. CHARACTER DIVERSITY
    # ============================================
    if length > 0:
        unique_chars = len(set(payload))
        features['char_diversity'] = unique_chars / length
        features['unique_char_count'] = unique_chars
    else:
        features['char_diversity'] = 0
        features['unique_char_count'] = 0
    
    # ============================================
    # 5. POSITION-BASED FEATURES
    # ============================================
    features['starts_with_space'] = 1 if payload.startswith(' ') else 0
    features['ends_with_space'] = 1 if payload.endswith(' ') else 0
    
    return features

# Load dataset
df = pd.read_csv('sqli_multiclass_dataset.csv')

print(f"\n✓ Loading dataset: {len(df)} samples")

# Extract features
print("\n✓ Extracting ultra-clean features...")
features_list = []
for idx, row in df.iterrows():
    if idx % 5000 == 0:
        print(f"  Processing {idx}/{len(df)}...")
    features = extract_ultra_clean_features(row['payload'])
    features_list.append(features)

# Create features dataframe
features_df = pd.DataFrame(features_list)

# Combine with labels
final_df = pd.concat([features_df, df[['sqli_class']]], axis=1)

# Save
final_df.to_csv('sqli_features_final.csv', index=False)

print("\n" + "="*80)
print("FEATURE EXTRACTION COMPLETE")
print("="*80)
print(f"✓ Total samples: {len(final_df)}")
print(f"✓ Total features: {len(features_df.columns)}")
print(f"✓ Saved to 'sqli_features_final.csv'")
print("\nFeatures extracted (NO SQL keywords):")
for i, col in enumerate(features_df.columns, 1):
    print(f"  {i}. {col}")
print("="*80)

STEP 2: FEATURE EXTRACTION (Ultra-Clean, No Leakage)

✓ Loading dataset: 30156 samples

✓ Extracting ultra-clean features...
  Processing 0/30156...
  Processing 5000/30156...
  Processing 10000/30156...
  Processing 15000/30156...
  Processing 20000/30156...
  Processing 25000/30156...
  Processing 30000/30156...

FEATURE EXTRACTION COMPLETE
✓ Total samples: 30156
✓ Total features: 27
✓ Saved to 'sqli_features_final.csv'

Features extracted (NO SQL keywords):
  1. num_spaces
  2. num_single_quotes
  3. num_double_quotes
  4. num_dashes
  5. num_semicolons
  6. num_parentheses
  7. num_equals
  8. num_asterisks
  9. num_percent
  10. num_commas
  11. num_slashes
  12. num_pipes
  13. num_ampersand
  14. num_dots
  15. num_underscores
  16. num_colons
  17. uppercase_ratio
  18. lowercase_ratio
  19. digit_ratio
  20. alpha_ratio
  21. special_char_ratio
  22. whitespace_ratio
  23. entropy
  24. char_diversity
  25. unique_char_count
  26. starts_with_space
  27. ends_with_space


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
from scipy.stats import ttest_rel, wilcoxon
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix)
from sklearn.utils.class_weight import compute_class_weight
from itertools import combinations

warnings.filterwarnings('ignore')

sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 10

print("=" * 80)
print("SQL INJECTION CLASSIFICATION - COMPREHENSIVE ANALYSIS")
print("=" * 80)

# ============================================================
# [1/9] LOAD AND PREPARE DATA
# ============================================================
print("\n[1/9] Loading data...")
df = pd.read_csv('sqli_features_final.csv')

if 'sqli_class' in df.columns:
    label_col = 'sqli_class'
elif 'sqli_class_v2' in df.columns:
    label_col = 'sqli_class_v2'
else:
    raise ValueError("Cannot find label column!")

X = df.drop(label_col, axis=1)
y = df[label_col]
feature_names = X.columns.tolist()

print(f"   Dataset: {len(df)} samples, {len(X.columns)} features")
print(f"   Classes: {y.nunique()} ({', '.join(y.unique())})")

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.3, random_state=42, stratify=y_encoded
)
print(f"   Training set: {len(X_train)} samples")
print(f"   Test set:     {len(X_test)} samples")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(class_weights))

# ============================================================
# [2/9] DEFINE MODELS
# ============================================================
print("\n[2/9] Defining models...")
models_config = {
    'Random Forest': RandomForestClassifier(
        n_estimators=100, max_depth=10, min_samples_split=10,
        min_samples_leaf=5, class_weight=class_weight_dict,
        random_state=42, n_jobs=1
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=100, learning_rate=0.05, max_depth=3,
        min_samples_split=10, subsample=0.8, random_state=42
    ),
    'SVM': SVC(
        C=0.5, gamma='scale', class_weight=class_weight_dict, random_state=42
    ),
    'Neural Network': MLPClassifier(
        hidden_layer_sizes=(64, 32), alpha=0.01, max_iter=300,
        early_stopping=True, random_state=42
    )
}
print(f"   {len(models_config)} models configured")

# ============================================================
# [3/9] CROSS-VALIDATION
# ============================================================
print("\n[3/9] Performing 5-fold cross-validation...")
cv_results = {}
cv_detailed_results = []

for model_name, model in models_config.items():
    print(f"   {model_name}...", end=" ", flush=True)
    cv_scores = cross_val_score(model, X_train_scaled, y_train,
                                cv=5, scoring='accuracy', n_jobs=1)
    cv_results[model_name] = {
        'scores': cv_scores,
        'mean':   cv_scores.mean(),
        'std':    cv_scores.std(),
        'min':    cv_scores.min(),
        'max':    cv_scores.max()
    }
    cv_detailed_results.append({
        'Model':              model_name,
        'Mean CV Accuracy':   cv_scores.mean(),
        'Standard Deviation': cv_scores.std(),
        'Min':                cv_scores.min(),
        'Max':                cv_scores.max()
    })
    print(f"Mean={cv_scores.mean():.4f}, Std={cv_scores.std():.4f}, "
          f"Min={cv_scores.min():.4f}, Max={cv_scores.max():.4f}")

cv_results_df = pd.DataFrame(cv_detailed_results)

print("\n" + "=" * 80)
print("CROSS-VALIDATION RESULTS")
print("=" * 80)
print(cv_results_df.to_string(index=False))

# ============================================================
# [4/9] TRAIN MODELS AND MEASURE INFERENCE TIME
# ============================================================
print("\n[4/9] Training models and measuring inference time...")
models      = {}
predictions = {}
train_times = {}
inference_times = {}
inference_times_per_sample = {}

for model_name, model in models_config.items():
    print(f"   {model_name}...", end=" ", flush=True)

    # --- Training time ---
    t0 = time.time()
    model.fit(X_train_scaled, y_train)
    train_times[model_name] = time.time() - t0

    models[model_name] = model

    # --- Inference time (100 runs for stability) ---
    n_runs = 100
    times = []
    for _ in range(n_runs):
        t0 = time.time()
        y_pred = model.predict(X_test_scaled)
        times.append(time.time() - t0)

    predictions[model_name]              = y_pred
    inference_times[model_name]          = np.mean(times)          # total for test set
    inference_times_per_sample[model_name] = (np.mean(times) / len(X_test)) * 1000  # ms/sample

    print(f"Train={train_times[model_name]:.2f}s | "
          f"Inference={inference_times[model_name]*1000:.2f}ms total | "
          f"{inference_times_per_sample[model_name]:.4f}ms/sample")

# ============================================================
# [5/9] TEST SET METRICS
# ============================================================
print("\n[5/9] Calculating test set metrics...")
results = []

for model_name, y_pred in predictions.items():
    acc        = accuracy_score(y_test, y_pred)
    prec_macro = precision_score(y_test, y_pred, average='macro', zero_division=0)
    rec_macro  = recall_score(y_test, y_pred, average='macro', zero_division=0)
    f1_macro   = f1_score(y_test, y_pred, average='macro', zero_division=0)

    results.append({
        'Model':                  model_name,
        'Mean CV Accuracy':       cv_results[model_name]['mean'],
        'Test Accuracy':          acc,
        'Test Precision (Macro)': prec_macro,
        'Test Recall (Macro)':    rec_macro,
        'Test F1 (Macro)':        f1_macro,
        'Train Time (s)':         round(train_times[model_name], 3),
        'Inference Time (ms)':    round(inference_times[model_name] * 1000, 3),
        'Inference/Sample (ms)':  round(inference_times_per_sample[model_name], 4)
    })

results_df = pd.DataFrame(results)
best_idx        = results_df['Test Accuracy'].idxmax()
best_model_name = results_df.loc[best_idx, 'Model']
best_model      = models[best_model_name]
best_pred       = predictions[best_model_name]

print("\n" + "=" * 80)
print("TEST SET RESULTS")
print("=" * 80)
print(results_df.to_string(index=False))
print(f"\nBest Model: {best_model_name} "
      f"(Accuracy={results_df.loc[best_idx,'Test Accuracy']:.4f})")

# ============================================================
# [6/9] STATISTICAL SIGNIFICANCE TESTING  ← NEW
# ============================================================
print("\n[6/9] Statistical significance testing (paired t-test + Wilcoxon)...")
print("   Comparing CV fold scores between all model pairs\n")

model_names_list = list(cv_results.keys())
stat_results = []

# Pairwise comparisons for all model combinations
pairs = list(combinations(model_names_list, 2))

for model_a, model_b in pairs:
    scores_a = cv_results[model_a]['scores']
    scores_b = cv_results[model_b]['scores']

    # Paired t-test
    t_stat, p_ttest = ttest_rel(scores_a, scores_b)

    # Wilcoxon signed-rank test (non-parametric alternative)
    try:
        w_stat, p_wilcoxon = wilcoxon(scores_a, scores_b)
    except ValueError:
        # Wilcoxon fails if all differences are zero
        w_stat, p_wilcoxon = np.nan, 1.0

    mean_diff = scores_a.mean() - scores_b.mean()
    significant_ttest    = "Yes" if p_ttest    < 0.05 else "No"
    significant_wilcoxon = "Yes" if p_wilcoxon < 0.05 else "No"

    stat_results.append({
        'Model A':             model_a,
        'Model B':             model_b,
        'Mean Diff (A-B)':     round(mean_diff, 6),
        't-statistic':         round(t_stat, 4),
        'p-value (t-test)':    round(p_ttest, 4),
        'Significant (t)':     significant_ttest,
        'W-statistic':         round(w_stat, 4) if not np.isnan(w_stat) else 'N/A',
        'p-value (Wilcoxon)':  round(p_wilcoxon, 4),
        'Significant (W)':     significant_wilcoxon
    })

    print(f"   {model_a} vs {model_b}:")
    print(f"     Mean diff = {mean_diff:+.6f} | "
          f"t={t_stat:.4f}, p={p_ttest:.4f} [{significant_ttest}] | "
          f"W p={p_wilcoxon:.4f} [{significant_wilcoxon}]")

stat_df = pd.DataFrame(stat_results)

print("\n" + "=" * 80)
print("STATISTICAL SIGNIFICANCE SUMMARY (alpha = 0.05)")
print("=" * 80)
print(stat_df[['Model A', 'Model B', 'Mean Diff (A-B)',
               'p-value (t-test)', 'Significant (t)',
               'p-value (Wilcoxon)', 'Significant (W)']].to_string(index=False))

# ============================================================
# [7/9] DEPLOYMENT EVALUATION  ← NEW
# ============================================================
print("\n[7/9] Practical deployment evaluation...")

deployment_data = []
for model_name in model_names_list:
    train_t   = train_times[model_name]
    infer_t   = inference_times[model_name] * 1000          # ms
    per_samp  = inference_times_per_sample[model_name]       # ms/sample
    throughput = 1000 / per_samp if per_samp > 0 else float('inf')  # samples/sec

    # Memory estimate (rough model size)
    import sys
    model_size_kb = sys.getsizeof(models[model_name]) / 1024

    # Real-time suitability: < 1ms per sample = suitable
    realtime = "Yes" if per_samp < 1.0 else "No"

    deployment_data.append({
        'Model':                  model_name,
        'Train Time (s)':         round(train_t, 3),
        'Inference Total (ms)':   round(infer_t, 3),
        'Per Sample (ms)':        round(per_samp, 4),
        'Throughput (samp/s)':    round(throughput, 1),
        'Real-Time Suitable':     realtime
    })

    print(f"   {model_name}:")
    print(f"     Training time:   {train_t:.3f}s")
    print(f"     Inference total: {infer_t:.3f}ms ({len(X_test)} samples)")
    print(f"     Per sample:      {per_samp:.4f}ms")
    print(f"     Throughput:      {throughput:.1f} samples/sec")
    print(f"     Real-time fit:   {realtime}")

deployment_df = pd.DataFrame(deployment_data)

print("\n" + "=" * 80)
print("DEPLOYMENT SUMMARY")
print("=" * 80)
print(deployment_df.to_string(index=False))

# ============================================================
# [8/9] FEATURE IMPORTANCE + PER-CLASS PERFORMANCE
# ============================================================
print(f"\n[8/9] Feature importance and per-class analysis ({best_model_name})...")

feature_importance_df = None

if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    feature_importance_df = pd.DataFrame({
        'Feature':          feature_names,
        'Importance Score': importances
    }).sort_values('Importance Score', ascending=False)
    top_10_features = feature_importance_df.head(10).copy()
    top_10_features.insert(0, 'Rank', range(1, 11))
    print(f"\n   Top 10 Features ({best_model_name}):")
    print(top_10_features.to_string(index=False))
else:
    print(f"   {best_model_name} has no native feature importances.")
    print("   Using Random Forest as backup...")
    rf_backup = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1)
    rf_backup.fit(X_train_scaled, y_train)
    importances = rf_backup.feature_importances_
    feature_importance_df = pd.DataFrame({
        'Feature':          feature_names,
        'Importance Score': importances
    }).sort_values('Importance Score', ascending=False)
    top_10_features = feature_importance_df.head(10).copy()
    top_10_features.insert(0, 'Rank', range(1, 11))
    print(top_10_features.to_string(index=False))

# Per-class performance
class_report = classification_report(
    y_test, best_pred,
    target_names=label_encoder.classes_,
    output_dict=True, zero_division=0
)
per_class_data = [{
    'Class':     c,
    'Precision': class_report[c]['precision'],
    'Recall':    class_report[c]['recall'],
    'F1-Score':  class_report[c]['f1-score'],
    'Support':   class_report[c]['support']
} for c in label_encoder.classes_]
per_class_df = pd.DataFrame(per_class_data)
print(f"\nPer-Class Performance ({best_model_name}):")
print(per_class_df.to_string(index=False))

# ============================================================
# SAVE ALL CSV FILES
# ============================================================
print("\nSaving CSV files...")
cv_results_df.to_csv('cv_results_detailed.csv', index=False)
results_df.to_csv('model_test_results.csv', index=False)
stat_df.to_csv('statistical_significance_results.csv', index=False)
deployment_df.to_csv('deployment_evaluation.csv', index=False)
per_class_df.to_csv('per_class_performance.csv', index=False)
if feature_importance_df is not None:
    feature_importance_df.to_csv('all_feature_importances.csv', index=False)
    top_10_features.to_csv('top_10_features.csv', index=False)

print("  Saved: cv_results_detailed.csv")
print("  Saved: model_test_results.csv")
print("  Saved: statistical_significance_results.csv")
print("  Saved: deployment_evaluation.csv")
print("  Saved: per_class_performance.csv")
print("  Saved: all_feature_importances.csv / top_10_features.csv")

# ============================================================
# [9/9] VISUALIZATIONS
# ============================================================
print("\n[9/9] Generating visualizations...")
colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']

# --- VIZ 1: CV Results Table ---
print("  [1/10] CV results table...")
fig, ax = plt.subplots(figsize=(14, 5))
ax.axis('off')
tdata = [[r['Model'], f"{r['Mean CV Accuracy']:.4f}",
          f"{r['Standard Deviation']:.4f}", f"{r['Min']:.4f}", f"{r['Max']:.4f}"]
         for _, r in cv_results_df.iterrows()]
tbl = ax.table(cellText=tdata,
               colLabels=['Model', 'Mean CV Accuracy', 'Std Dev', 'Min', 'Max'],
               cellLoc='center', loc='center',
               colWidths=[0.25, 0.20, 0.20, 0.175, 0.175])
tbl.auto_set_font_size(False); tbl.set_fontsize(12); tbl.scale(1, 3)
for i in range(5):
    tbl[(0,i)].set_facecolor('#2c3e50')
    tbl[(0,i)].set_text_props(weight='bold', color='white', size=13)
for i in range(1, len(tdata)+1):
    fc = '#ecf0f1' if i % 2 == 0 else '#ffffff'
    for j in range(5):
        tbl[(i,j)].set_facecolor(fc)
        tbl[(i,j)].set_text_props(weight='bold', size=11)
plt.title('Cross-Validation Results (5-Fold)', fontsize=16, fontweight='bold', pad=20)
plt.savefig('01_cv_results_table.png', dpi=300, bbox_inches='tight'); plt.close()
print("     Saved: 01_cv_results_table.png")

# --- VIZ 2: CV Accuracy Error Bars ---
print("  [2/10] CV accuracy with error bars...")
fig, ax = plt.subplots(figsize=(12, 7))
means = cv_results_df['Mean CV Accuracy'].tolist()
stds  = cv_results_df['Standard Deviation'].tolist()
bars  = ax.bar(model_names_list, means, yerr=stds, capsize=12,
               color=colors, alpha=0.85, edgecolor='black', linewidth=2,
               error_kw={'linewidth': 2.5, 'ecolor': 'black', 'alpha': 0.7})
ax.set_ylabel('Accuracy', fontsize=14, fontweight='bold')
ax.set_xlabel('Model', fontsize=14, fontweight='bold')
ax.set_title('Cross-Validation Accuracy (Mean ± Std)', fontsize=16, fontweight='bold', pad=20)
ax.set_ylim([0, 1.1]); ax.grid(axis='y', alpha=0.3, linestyle='--')
plt.xticks(rotation=45, ha='right', fontsize=12, fontweight='bold')
for bar, mean, std in zip(bars, means, stds):
    ax.text(bar.get_x()+bar.get_width()/2., bar.get_height()+std+0.02,
            f'{mean:.4f}\n±{std:.4f}', ha='center', va='bottom',
            fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig('02_cv_accuracy_comparison.png', dpi=300, bbox_inches='tight'); plt.close()
print("     Saved: 02_cv_accuracy_comparison.png")

# --- VIZ 3: CV Range ---
print("  [3/10] CV min-max range...")
fig, ax = plt.subplots(figsize=(12, 7))
mins = cv_results_df['Min'].tolist()
maxs = cv_results_df['Max'].tolist()
x    = np.arange(len(model_names_list))
bars = ax.bar(x, means, 0.6, color=colors, alpha=0.85, edgecolor='black', linewidth=2)
ax.errorbar(x, means,
            yerr=[[m-mn for m,mn in zip(means,mins)],
                  [mx-m  for m,mx in zip(means,maxs)]],
            fmt='none', ecolor='black', capsize=12, capthick=2.5, linewidth=2.5)
ax.set_ylabel('Accuracy', fontsize=14, fontweight='bold')
ax.set_xlabel('Model', fontsize=14, fontweight='bold')
ax.set_title('CV Accuracy Range (Min to Max)', fontsize=16, fontweight='bold', pad=20)
ax.set_xticks(x); ax.set_xticklabels(model_names_list, rotation=45, ha='right',
                                      fontsize=12, fontweight='bold')
ax.set_ylim([0, 1.1]); ax.grid(axis='y', alpha=0.3, linestyle='--')
for i, (bar, mean, mn, mx) in enumerate(zip(bars, means, mins, maxs)):
    ax.text(bar.get_x()+bar.get_width()/2., mx+0.02,
            f'Max:{mx:.3f}\nMean:{mean:.3f}\nMin:{mn:.3f}',
            ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.savefig('03_cv_range_visualization.png', dpi=300, bbox_inches='tight'); plt.close()
print("     Saved: 03_cv_range_visualization.png")

# --- VIZ 4: Box Plot ---
print("  [4/10] CV box plot...")
fig, ax = plt.subplots(figsize=(12, 7))
bp = ax.boxplot([cv_results[m]['scores'] for m in model_names_list],
                labels=model_names_list, patch_artist=True, notch=True, showmeans=True,
                boxprops=dict(linewidth=2),
                whiskerprops=dict(linewidth=2, color='black'),
                capprops=dict(linewidth=2, color='black'),
                medianprops=dict(linewidth=2.5, color='red'),
                meanprops=dict(marker='D', markerfacecolor='yellow',
                               markeredgecolor='black', markersize=8))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color); patch.set_alpha(0.7)
ax.set_ylabel('Accuracy', fontsize=14, fontweight='bold')
ax.set_title('CV Score Distribution (Box Plot)', fontsize=16, fontweight='bold', pad=20)
ax.grid(axis='y', alpha=0.3, linestyle='--')
plt.xticks(rotation=45, ha='right', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('04_cv_boxplot.png', dpi=300, bbox_inches='tight'); plt.close()
print("     Saved: 04_cv_boxplot.png")

# --- VIZ 5: Confusion Matrix ---
print(f"  [5/10] Confusion matrix ({best_model_name})...")
fig, ax = plt.subplots(figsize=(10, 8))
cm = confusion_matrix(y_test, best_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_,
            ax=ax, linewidths=2, linecolor='white',
            annot_kws={'size': 14, 'weight': 'bold'})
ax.set_ylabel('True Label', fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted Label', fontsize=14, fontweight='bold')
ax.set_title(f'Confusion Matrix - {best_model_name}', fontsize=16, fontweight='bold', pad=20)
plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=11, fontweight='bold')
plt.setp(ax.get_yticklabels(), rotation=0, fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('05_best_model_confusion_matrix.png', dpi=300, bbox_inches='tight'); plt.close()
print("     Saved: 05_best_model_confusion_matrix.png")

# --- VIZ 6: Statistical Significance Heatmap  ← NEW ---
print("  [6/10] Statistical significance heatmap...")
n = len(model_names_list)
pval_matrix = np.ones((n, n))
sig_matrix  = np.zeros((n, n))

for i, model_a in enumerate(model_names_list):
    for j, model_b in enumerate(model_names_list):
        if i != j:
            _, p = ttest_rel(cv_results[model_a]['scores'],
                             cv_results[model_b]['scores'])
            pval_matrix[i, j] = p
            sig_matrix[i, j]  = 1 if p < 0.05 else 0

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# p-value heatmap
sns.heatmap(pval_matrix, annot=True, fmt='.4f', cmap='RdYlGn_r',
            xticklabels=model_names_list, yticklabels=model_names_list,
            ax=axes[0], vmin=0, vmax=0.1, linewidths=1, linecolor='white',
            annot_kws={'size': 11, 'weight': 'bold'})
axes[0].set_title('Paired t-test p-values\n(green = not significant, red = significant)',
                  fontsize=13, fontweight='bold', pad=15)
plt.setp(axes[0].get_xticklabels(), rotation=45, ha='right', fontsize=10)
plt.setp(axes[0].get_yticklabels(), rotation=0, fontsize=10)

# Significance heatmap (0/1)
sig_labels = np.where(pval_matrix < 0.05, 'p<0.05\nSig', 'p≥0.05\nNot Sig')
np.fill_diagonal(sig_labels, '—')
sig_plot = np.where(pval_matrix < 0.05, 1.0, 0.0)
np.fill_diagonal(sig_plot, 0.5)
sns.heatmap(sig_plot, annot=sig_labels, fmt='', cmap='RdYlGn',
            xticklabels=model_names_list, yticklabels=model_names_list,
            ax=axes[1], vmin=0, vmax=1, linewidths=1, linecolor='white',
            annot_kws={'size': 10, 'weight': 'bold'}, cbar=False)
axes[1].set_title('Significance Summary\n(alpha = 0.05)',
                  fontsize=13, fontweight='bold', pad=15)
plt.setp(axes[1].get_xticklabels(), rotation=45, ha='right', fontsize=10)
plt.setp(axes[1].get_yticklabels(), rotation=0, fontsize=10)

plt.suptitle('Statistical Significance Testing — Paired t-test on CV Fold Scores',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('06_statistical_significance.png', dpi=300, bbox_inches='tight'); plt.close()
print("     Saved: 06_statistical_significance.png")

# --- VIZ 7: Deployment Comparison  ← NEW ---
print("  [7/10] Deployment evaluation chart...")
fig, axes = plt.subplots(1, 3, figsize=(18, 7))

# Plot 1: Training time
axes[0].bar(deployment_df['Model'], deployment_df['Train Time (s)'],
            color=colors, alpha=0.85, edgecolor='black', linewidth=1.5)
axes[0].set_title('Training Time (seconds)', fontsize=13, fontweight='bold', pad=12)
axes[0].set_ylabel('Seconds', fontsize=12, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3, linestyle='--')
plt.setp(axes[0].get_xticklabels(), rotation=45, ha='right', fontsize=10, fontweight='bold')
for bar, val in zip(axes[0].patches, deployment_df['Train Time (s)']):
    axes[0].text(bar.get_x()+bar.get_width()/2., bar.get_height()+0.01,
                 f'{val:.3f}s', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Plot 2: Inference time per sample
axes[1].bar(deployment_df['Model'], deployment_df['Per Sample (ms)'],
            color=colors, alpha=0.85, edgecolor='black', linewidth=1.5)
axes[1].set_title('Inference Time per Sample (ms)', fontsize=13, fontweight='bold', pad=12)
axes[1].set_ylabel('Milliseconds', fontsize=12, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3, linestyle='--')
plt.setp(axes[1].get_xticklabels(), rotation=45, ha='right', fontsize=10, fontweight='bold')
for bar, val in zip(axes[1].patches, deployment_df['Per Sample (ms)']):
    axes[1].text(bar.get_x()+bar.get_width()/2., bar.get_height()+0.00002,
                 f'{val:.4f}ms', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Plot 3: Throughput
axes[2].bar(deployment_df['Model'], deployment_df['Throughput (samp/s)'],
            color=colors, alpha=0.85, edgecolor='black', linewidth=1.5)
axes[2].set_title('Throughput (samples/second)', fontsize=13, fontweight='bold', pad=12)
axes[2].set_ylabel('Samples/sec', fontsize=12, fontweight='bold')
axes[2].grid(axis='y', alpha=0.3, linestyle='--')
plt.setp(axes[2].get_xticklabels(), rotation=45, ha='right', fontsize=10, fontweight='bold')
for bar, val in zip(axes[2].patches, deployment_df['Throughput (samp/s)']):
    axes[2].text(bar.get_x()+bar.get_width()/2., bar.get_height()*1.01,
                 f'{val:,.0f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.suptitle('Practical Deployment Evaluation — Computational Cost Analysis',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('07_deployment_evaluation.png', dpi=300, bbox_inches='tight'); plt.close()
print("     Saved: 07_deployment_evaluation.png")

# --- VIZ 8: Top 10 Features ---
if feature_importance_df is not None:
    print("  [8/10] Top 10 features chart...")
    fig, ax = plt.subplots(figsize=(12, 8))
    y_pos   = np.arange(10)
    feats   = top_10_features['Feature'].tolist()
    imps    = top_10_features['Importance Score'].tolist()
    bars    = ax.barh(y_pos, imps,
                      color=plt.cm.viridis(np.linspace(0.3, 0.9, 10)),
                      alpha=0.85, edgecolor='black', linewidth=1.5)
    ax.set_yticks(y_pos); ax.set_yticklabels(feats, fontsize=11, fontweight='bold')
    ax.set_xlabel('Importance Score', fontsize=13, fontweight='bold')
    ax.set_title(f'Top 10 Feature Importances ({best_model_name})',
                 fontsize=16, fontweight='bold', pad=20)
    ax.grid(axis='x', alpha=0.3, linestyle='--')
    for i, (bar, imp) in enumerate(zip(bars, imps)):
        ax.text(bar.get_width()+0.001, bar.get_y()+bar.get_height()/2.,
                f'{imp:.6f}', ha='left', va='center', fontsize=9, fontweight='bold')
        ax.text(-0.002, bar.get_y()+bar.get_height()/2., f'#{i+1}',
                ha='right', va='center', fontsize=10, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7))
    plt.tight_layout()
    plt.savefig('08_top_10_features_chart.png', dpi=300, bbox_inches='tight'); plt.close()
    print("     Saved: 08_top_10_features_chart.png")

# --- VIZ 9: Per-Class Performance ---
print(f"  [9/10] Per-class performance ({best_model_name})...")
fig, ax = plt.subplots(figsize=(12, 8))
classes    = per_class_df['Class'].tolist()
f1_scores  = per_class_df['F1-Score'].tolist()
support    = per_class_df['Support'].tolist()
y_pos      = np.arange(len(classes))
bars       = ax.barh(y_pos, f1_scores, color='steelblue', alpha=0.85,
                     edgecolor='black', linewidth=1.5)
ax.set_yticks(y_pos); ax.set_yticklabels(classes, fontsize=12, fontweight='bold')
ax.set_xlabel('F1-Score', fontsize=13, fontweight='bold')
ax.set_title(f'Per-Class F1-Scores — {best_model_name}',
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlim([0, 1.1]); ax.grid(axis='x', alpha=0.3, linestyle='--')
for bar, score, sup in zip(bars, f1_scores, support):
    ax.text(bar.get_width()+0.02, bar.get_y()+bar.get_height()/2.,
            f'{score:.4f} (n={int(sup)})', ha='left', va='center',
            fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig('09_per_class_performance.png', dpi=300, bbox_inches='tight'); plt.close()
print("     Saved: 09_per_class_performance.png")

# --- VIZ 10: Deployment Summary Table ---
print("  [10/10] Deployment summary table...")
fig, ax = plt.subplots(figsize=(16, 5))
ax.axis('off')
tdata = [[r['Model'], f"{r['Train Time (s)']:.3f}s",
          f"{r['Inference Total (ms)']:.3f}ms",
          f"{r['Per Sample (ms)']:.4f}ms",
          f"{r['Throughput (samp/s)']:,.0f}",
          r['Real-Time Suitable']]
         for _, r in deployment_df.iterrows()]
tbl = ax.table(
    cellText=tdata,
    colLabels=['Model', 'Train Time', 'Inference Total',
               'Per Sample', 'Throughput (s/s)', 'Real-Time Ready'],
    cellLoc='center', loc='center',
    colWidths=[0.20, 0.15, 0.18, 0.15, 0.18, 0.14])
tbl.auto_set_font_size(False); tbl.set_fontsize(11); tbl.scale(1, 3)
for i in range(6):
    tbl[(0,i)].set_facecolor('#2c3e50')
    tbl[(0,i)].set_text_props(weight='bold', color='white', size=12)
for i in range(1, len(tdata)+1):
    fc = '#ecf0f1' if i % 2 == 0 else '#ffffff'
    for j in range(6):
        tbl[(i,j)].set_facecolor(fc)
        tbl[(i,j)].set_text_props(weight='bold', size=11)
    # Colour the Real-Time column
    rt_val = tdata[i-1][5]
    tbl[(i,5)].set_facecolor('#d5f5e3' if rt_val == 'Yes' else '#fde8e8')
plt.title('Practical Deployment Evaluation Summary',
          fontsize=16, fontweight='bold', pad=20)
plt.savefig('10_deployment_summary_table.png', dpi=300, bbox_inches='tight'); plt.close()
print("     Saved: 10_deployment_summary_table.png")

# ============================================================
# FINAL SUMMARY
# ============================================================
print("\n" + "=" * 80)
print("COMPLETE!")
print("=" * 80)
print("\nCSV Files:")
print("  cv_results_detailed.csv             - CV with Mean, Std, Min, Max")
print("  model_test_results.csv              - Test set performance + timing")
print("  statistical_significance_results.csv - Paired t-test + Wilcoxon")
print("  deployment_evaluation.csv           - Training/inference/throughput")
print("  per_class_performance.csv           - Per-class metrics")
print("  all_feature_importances.csv         - All 27 features ranked")
print("  top_10_features.csv                 - Top 10 features")
print("\nVisualizations (300 DPI):")
print("  01_cv_results_table.png")
print("  02_cv_accuracy_comparison.png")
print("  03_cv_range_visualization.png")
print("  04_cv_boxplot.png")
print("  05_best_model_confusion_matrix.png")
print("  06_statistical_significance.png     ← NEW")
print("  07_deployment_evaluation.png        ← NEW")
print("  08_top_10_features_chart.png")
print("  09_per_class_performance.png")
print("  10_deployment_summary_table.png     ← NEW")
print("=" * 80)

SQL INJECTION CLASSIFICATION - COMPREHENSIVE ANALYSIS

[1/9] Loading data...
   Dataset: 30156 samples, 27 features
   Classes: 7 (other_sqli, benign, piggy_backed, error_based, time_based_blind, union_based, boolean_blind)
   Training set: 21109 samples
   Test set:     9047 samples

[2/9] Defining models...
   4 models configured

[3/9] Performing 5-fold cross-validation...
   Random Forest... Mean=0.9865, Std=0.0023, Min=0.9820, Max=0.9886
   Gradient Boosting... Mean=0.9891, Std=0.0010, Min=0.9877, Max=0.9905
   SVM... Mean=0.9869, Std=0.0020, Min=0.9848, Max=0.9905
   Neural Network... Mean=0.9953, Std=0.0014, Min=0.9931, Max=0.9967

CROSS-VALIDATION RESULTS
            Model  Mean CV Accuracy  Standard Deviation      Min      Max
    Random Forest          0.986451            0.002298 0.981999 0.988631
Gradient Boosting          0.989104            0.001038 0.987684 0.990526
              SVM          0.986878            0.002007 0.984841 0.990526
   Neural Network          0.995